In [1]:
!pip install pandas numpy scikit-learn matplotlib joblib

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

quiz_avg       = np.round(np.random.uniform(20, 100, n), 1)
assignment_avg = np.round(np.random.uniform(20, 100, n), 1)
exam_avg       = np.round(np.random.uniform(20, 100, n), 1)
attendance_pct = np.round(np.random.uniform(30, 100, n), 1)
prev_gpa       = np.round(np.random.uniform(30, 100, n), 1)
study_hours    = np.round(np.random.uniform(0.5, 8.0, n), 1)

# Weights explain how much each feature contributes to final score
# Exam avg matters most (40%) because midterms are closest to finals
# prev_gpa second (20%) because past performance predicts future
# study_hours multiplied by 10 to bring it to same scale as others
final_score = (
    0.40 * exam_avg +
    0.20 * prev_gpa +
    0.15 * quiz_avg +
    0.10 * assignment_avg +
    0.10 * attendance_pct +
    0.05 * (study_hours * 10) +
    np.random.normal(0, 5, n)
)

final_score = np.clip(np.round(final_score, 1), 0, 100)

df = pd.DataFrame({
    'quiz_avg':       quiz_avg,
    'assignment_avg': assignment_avg,
    'exam_avg':       exam_avg,
    'attendance_pct': attendance_pct,
    'prev_gpa':       prev_gpa,
    'study_hours':    study_hours,
    'final_score':    final_score
})

df.to_csv('students_forecasting.csv', index=False)

print(f'Dataset: {df.shape[0]} students, {df.shape[1]} columns')
df.head(8)

Dataset: 500 students, 7 columns


,quiz_avg,assignment_avg,exam_avg,attendance_pct,prev_gpa,study_hours,final_score
0,50.0,75.9,34.8,66.3,48.3,6.7,53.9
1,96.1,62.9,63.4,63.5,47.3,6.2,68.1
2,78.6,44.8,89.8,31.8,93.4,4.8,79.9
3,67.9,85.1,78.6,53.9,47.5,7.7,62.0
4,32.5,74.8,84.5,56.6,49.0,2.0,68.7
5,32.5,33.0,72.7,57.9,83.2,1.3,61.6
6,24.6,92.9,75.4,70.6,61.5,6.9,64.1
7,89.3,85.8,87.9,67.4,84.4,3.8,83.4


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_cols = ['quiz_avg', 'assignment_avg', 'exam_avg',
                'attendance_pct', 'prev_gpa', 'study_hours']

X = df[feature_cols]
y = df['final_score']

scaler2  = StandardScaler()
X_scaled = scaler2.fit_transform(X)

# No stratify here — stratify only works for classification
# For regression the target is continuous numbers so no stratify
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

rf2 = RandomForestRegressor(n_estimators=100, random_state=42)
rf2.fit(X_train, y_train)

preds = rf2.predict(X_test)

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2   = r2_score(y_test, preds)

print('Model Evaluation:')
print(f'  MAE  : {mae:.2f} marks  (avg prediction is off by this many marks)')
print(f'  RMSE : {rmse:.2f} marks')
print(f'  R2   : {r2:.4f}         (closer to 1.0 = better fit)')

Model Evaluation:
  MAE  : 4.77 marks  (avg prediction is off by this many marks)
  RMSE : 5.81 marks
  R2   : 0.7722         (closer to 1.0 = better fit)


In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_cols = ['quiz_avg', 'assignment_avg', 'exam_avg',
                'attendance_pct', 'prev_gpa', 'study_hours']

X = df[feature_cols]
y = df['final_score']

# Scale features
scaler2 = StandardScaler()
X_scaled = scaler2.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Note: No stratify here because y is continuous numbers not categories

# Train
rf2 = RandomForestRegressor(n_estimators=100, random_state=42)
rf2.fit(X_train, y_train)

preds = rf2.predict(X_test)

# Evaluation metrics for regression
mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2   = r2_score(y_test, preds)

print('Model Evaluation:')
print(f'  MAE  (Mean Absolute Error)  : {mae:.2f} marks')
print(f'  RMSE (Root Mean Sq Error)   : {rmse:.2f} marks')
print(f'  R2   (Accuracy of fit)      : {r2:.4f}')
print()
print('How to read these:')
print(f'  MAE {mae:.1f} means on average the prediction is off by {mae:.1f} marks')
print(f'  R2 {r2:.2f} means the model explains {r2*100:.0f}% of score variation')

Model Evaluation:
  MAE  (Mean Absolute Error)  : 4.77 marks
  RMSE (Root Mean Sq Error)   : 5.81 marks
  R2   (Accuracy of fit)      : 0.7722

How to read these:
  MAE 4.8 means on average the prediction is off by 4.8 marks
  R2 0.77 means the model explains 77% of score variation


In [5]:
import joblib, os

os.makedirs('models', exist_ok=True)

joblib.dump(rf2,     'models/forecast_model.pkl')
joblib.dump(scaler2, 'models/forecast_scaler.pkl')

print('Saved:')
print('  models/forecast_model.pkl')
print('  models/forecast_scaler.pkl')
print()
print('No label encoder needed for regression')

Saved:
  models/forecast_model.pkl
  models/forecast_scaler.pkl

No label encoder needed for regression


In [6]:
def get_suggestions(student: dict) -> list:
    suggestions = []

    if student['quiz_avg'] < 55:
        suggestions.append('Practice more quizzes — aim for weekly revision')

    if student['assignment_avg'] < 55:
        suggestions.append('Complete all assignments on time for consistent marks')

    if student['exam_avg'] < 55:
        suggestions.append('Focus on exam preparation — solve previous year papers')

    if student['attendance_pct'] < 75:
        suggestions.append('Improve attendance — aim for at least 75%')

    if student['prev_gpa'] < 50:
        suggestions.append('Revise core concepts from previous semester')

    if student['study_hours'] < 1.5:
        suggestions.append('Increase daily study hours — aim for at least 2-3 hours per day')

    return suggestions if suggestions else ['Keep up the great work!']


# Test
test = {
    'quiz_avg': 45.0, 'assignment_avg': 72.0,
    'exam_avg': 60.0, 'attendance_pct': 55.0,
    'prev_gpa': 65.0, 'study_hours': 1.0
}

print('Suggestions for test student:')
for s in get_suggestions(test):
    print(f'  → {s}')

Suggestions for test student:
  → Practice more quizzes — aim for weekly revision
  → Improve attendance — aim for at least 75%
  → Increase daily study hours — aim for at least 2-3 hours per day


In [7]:
import joblib
import numpy as np

_model2  = joblib.load('models/forecast_model.pkl')
_scaler2 = joblib.load('models/forecast_scaler.pkl')


def predict_score(student: dict) -> dict:
    features = np.array([[
        student['quiz_avg'],
        student['assignment_avg'],
        student['exam_avg'],
        student['attendance_pct'],
        student['prev_gpa'],
        student['study_hours']
    ]])

    scaled          = _scaler2.transform(features)
    predicted_score = round(float(_model2.predict(scaled)[0]), 1)
    predicted_score = max(0, min(100, predicted_score))

    if predicted_score >= 70:   grade = 'Distinction'
    elif predicted_score >= 60: grade = 'First Class'
    elif predicted_score >= 50: grade = 'Second Class'
    elif predicted_score >= 40: grade = 'Pass'
    else:                       grade = 'At Risk of Failing'

    if predicted_score < 50:
        improvement = f'Need +{round(50 - predicted_score, 1)} marks to pass'
    elif predicted_score < 70:
        improvement = f'Need +{round(70 - predicted_score, 1)} marks for distinction'
    else:
        improvement = 'On track for distinction'

    return {
        'predicted_score': predicted_score,
        'grade':           grade,
        'improvement':     improvement,
        'suggestions':     get_suggestions(student)
    }


# ── Test 1: struggling ───────────────────────────────────────────────
struggling = {
    'quiz_avg': 35.0, 'assignment_avg': 40.0, 'exam_avg': 38.0,
    'attendance_pct': 50.0, 'prev_gpa': 42.0, 'study_hours': 0.5
}

# ── Test 2: average ──────────────────────────────────────────────────
average = {
    'quiz_avg': 60.0, 'assignment_avg': 65.0, 'exam_avg': 62.0,
    'attendance_pct': 75.0, 'prev_gpa': 63.0, 'study_hours': 2.0
}

# ── Test 3: strong ───────────────────────────────────────────────────
strong = {
    'quiz_avg': 88.0, 'assignment_avg': 85.0, 'exam_avg': 90.0,
    'attendance_pct': 95.0, 'prev_gpa': 87.0, 'study_hours': 5.0
}

for name, student in [('Struggling', struggling), ('Average', average), ('Strong', strong)]:
    result = predict_score(student)
    print(f'── {name} student ──')
    print(f"  Predicted score : {result['predicted_score']}/100")
    print(f"  Grade           : {result['grade']}")
    print(f"  Improvement     : {result['improvement']}")
    print(f"  Suggestions     :")
    for s in result['suggestions']:
        print(f"    → {s}")
    print()

── Struggling student ──
  Predicted score : 41.5/100
  Grade           : Pass
  Improvement     : Need +8.5 marks to pass
  Suggestions     :
    → Practice more quizzes — aim for weekly revision
    → Complete all assignments on time for consistent marks
    → Focus on exam preparation — solve previous year papers
    → Improve attendance — aim for at least 75%
    → Revise core concepts from previous semester
    → Increase daily study hours — aim for at least 2-3 hours per day

── Average student ──
  Predicted score : 63.4/100
  Grade           : First Class
  Improvement     : Need +6.6 marks for distinction
  Suggestions     :
    → Keep up the great work!

── Strong student ──
  Predicted score : 82.5/100
  Grade           : Distinction
  Improvement     : On track for distinction
  Suggestions     :
    → Keep up the great work!



c:\anaconda\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\anaconda\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\anaconda\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
